# Simulación de escenarios críticos (workshop de la semana 5)
**Proyecto:** Predicción de matrícula y continuidad estudiantil · Unidad Educativa LEMAS
**Integrantes:** Guillermo Granizo y José Ulloa · Maestría en Inteligencia Artificial (UEES)

**Pregunta guía:** ¿hasta qué nivel de perturbación la lista sigue siendo útil, y qué hace el sistema cuando deja de serlo?

| Escenario del workshop | Versión realista para LEMAS | Qué se mide |
|---|---|---|
| Ruido en la entrada | Errores de digitación en el Excel: se invierte el pago tardío o el tipo de reserva, o los atrasos se corren en uno, con probabilidad 1 % a 20 %. Ruido gaussiano solo en el promedio (variable continua de la logística híbrida) | Caída del Lift@k |
| Datos faltantes | Del 10 % al 50 % de una señal llega vacía, una señal por vez | Degradación por señal |
| Desbalance extremo | Se quitan familias con caso hasta una tasa del 5 %, 3 % y 1 % | Lift@k, Recall@k y PR-AUC |
| Cambio de distribución | La logística híbrida se entrena con una sede (o con los no becados) y se prueba con la otra (o con los becados) | PR-AUC, ROC-AUC y calibración |
| Feature clave caída | Una señal llega constante: toda en cero o toda «en plazo» | Lift@k y Recall@k |

**Reglas:** se perturba solo **C4**, con el sistema ya elegido (la regla D2 y la logística híbrida entrenada con C2 + C3, como en la Fase 2). **C5 no interviene.** No se usa F1: la lista ordena con capacidad fija (k = 165 familias), no clasifica con un umbral. Cada nivel se repite varias veces con otra perturbación y otro sorteo de empates; se guardan solo **agregados** (medias y percentiles), nunca datos individuales.

In [ ]:
# @title 0 · Preparar el entorno (ejecutar primero)
# Si el repositorio ya está en GitHub, pegue su URL; si no, se pedirá subir el .zip.
REPO_URL = "https://github.com/ggranizo2507/prediccion-matricula-lemas.git"  # @param {type:"string"}

import importlib.util, os, subprocess, sys, zipfile
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    RAIZ = Path("/content/prediccion-matricula-lemas")
    if not RAIZ.exists():
        if REPO_URL:
            subprocess.run(["git", "clone", "-q", REPO_URL, str(RAIZ)], check=True)
        else:
            from google.colab import files
            print("Suba el archivo prediccion-matricula-lemas.zip")
            nombre = next(iter(files.upload()))
            zipfile.ZipFile(nombre).extractall("/content")
else:
    RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))
REQUERIDOS = {'pandas': 'pandas', 'sklearn': 'scikit-learn', 'yaml': 'PyYAML',
              'optuna': 'optuna', 'matplotlib': 'matplotlib'}  # módulo -> paquete pip
faltan = [pip for modulo, pip in REQUERIDOS.items() if importlib.util.find_spec(modulo) is None]
if faltan:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltan], check=True)
print("Entorno:", "Google Colab" if EN_COLAB else "local")

In [ ]:
# @title Cargar la base
FUENTE = "real"  # @param ["sintetica", "real"]
SUBIR_DE_NUEVO = False  # @param {type:"boolean"}

import json, logging, warnings
warnings.filterwarnings("ignore", message=".*IProgress.*")
import numpy as np
import pandas as pd
from IPython.display import display
from src.data_processing import construir_dataset, limpiar_base
from src.utils import cargar_config, configurar_logging, obtener_base_seud

configurar_logging(logging.WARNING)
config = cargar_config()
SEMILLA = config["proyecto"]["semilla"]
MIN = config["privacidad"]["min_celda"]
REPETICIONES = config["escenarios"]["repeticiones"]
UMBRAL = config["escenarios"]["umbral_lift"]
ruta = RAIZ / config["rutas"]["base_real" if FUENTE == "real" else "base_sintetica"]
crudo = (obtener_base_seud(ruta, EN_COLAB, SUBIR_DE_NUEVO) if FUENTE == "real"
         else pd.read_csv(ruta, dtype=str))
dataset, _ = construir_dataset(limpiar_base(crudo, config), config)
print(f"Fuente: {FUENTE} | estudiantes modelables: {len(dataset)} | repeticiones por nivel: {REPETICIONES}")
if FUENTE == "sintetica":
    print("⚠️ Datos SINTÉTICOS: sirven para probar el código, no describen a LEMAS.")

In [ ]:
# @title Sistema ya elegido y cohorte C4 (sin C5)
from src import diagnostico as dg
from src import escenarios as es
from src.evaluate import calcular_k_por_sede, consolidar_familias

n = dg.numero
MET = RAIZ / config["rutas"]["metricas"]; MET.mkdir(parents=True, exist_ok=True)
FIG = RAIZ / config["rutas"]["figuras"]; FIG.mkdir(parents=True, exist_ok=True)
DPI = config["figuras"]["dpi"]

decision = json.loads((MET / f"decision_c1_{FUENTE}.json").read_text(encoding="utf-8"))
COHORTES = config["auditoria"]["configuraciones_entrenamiento"]["con_C1" if decision["incluir_C1"] else "sin_C1"]
K = decision["k_por_sede"]
PARAMS = dg.mejores_parametros(pd.read_csv(MET / f"optuna_historial_{FUENTE}.csv"))["hibrido"]

usadas = dg.sin_prueba(dataset)                   # C5 fuera
ENTRENAMIENTO = usadas[usadas["cohorte"].isin(COHORTES)]
C4 = usadas[usadas["rol"] == "seleccion"]
HIBRIDO = es.entrenar_hibrido(ENTRENAMIENTO, PARAMS, SEMILLA)
SISTEMAS = {es.REGLA: es.puntuador_d2, es.HIBRIDO: es.puntuador_modelo(HIBRIDO)}

# Comprobación: sin perturbar y con la semilla fija se reproducen las cifras de la Fase 2 en C4
COMPROBACION = {nombre: es.medir(C4, f, K, SEMILLA) for nombre, f in SISTEMAS.items()}
print("Entrenamiento:", COHORTES, "| Perturbaciones sobre: C4 | C5: no se usa | k por sede:", K)
print("Hiperparámetros de la logística híbrida (Fase 2):", PARAMS)
display(pd.DataFrame(COMPROBACION).T.round(4))
ruta_seleccion = MET / f"seleccion_C4_{FUENTE}.csv"
if ruta_seleccion.exists():
    fase2 = pd.read_csv(ruta_seleccion).set_index("candidato")["lift_k"]
    print("Lift@k en la Fase 2:", {k: round(float(v), 4) for k, v in fase2.items()
                                    if k.startswith(("D2", "Logística híbrida"))})

## 1 y 2. Ruido y datos faltantes
Las señales de la regla D2 son discretas: un error realista es una celda mal digitada, no un ruido gaussiano. El ruido gaussiano se aplica solo al promedio, la variable continua que usa la logística híbrida. En los faltantes, un pago tardío vacío deja al estudiante sin esa prioridad y una reserva o un atraso vacíos se leen como cero (la regla) o se imputan (el modelo).

In [ ]:
# @title Errores de registro, ruido en el promedio y datos faltantes
RUIDO = es.escenario_ruido(C4, SISTEMAS, K, SEMILLA, REPETICIONES)
FALTANTES = es.escenario_faltantes(C4, SISTEMAS, K, SEMILLA, REPETICIONES)
columnas = ["variable", "sistema", "nivel", "lift_k", "lift_k_p05", "lift_k_p95", "recall_k", "pr_auc"]
display(RUIDO[columnas].round(3))
display(FALTANTES[columnas].round(3))

## 3 y 5. Desbalance extremo y señal caída
En el desbalance se quitan familias con caso (con todos sus estudiantes) hasta la tasa indicada, como en un año con muy pocas familias que no se matriculan. El k sigue siendo el aprobado. En la señal caída, la perturbación es fija y se repite solo el sorteo de empates.

In [ ]:
# @title Desbalance extremo y señal caída
DESBALANCE = es.escenario_desbalance(C4, SISTEMAS, K, SEMILLA, REPETICIONES)
CAIDA = es.escenario_caida(C4, SISTEMAS, K, SEMILLA, REPETICIONES)
display(DESBALANCE[["sistema", "nivel", "tasa_real", "lift_k", "lift_k_p05", "lift_k_p95", "recall_k", "pr_auc"]].round(3))
display(CAIDA[["variable", "sistema", "lift_k", "lift_k_p05", "lift_k_p95", "recall_k", "pr_auc"]].round(3))

## 4. Cambio de distribución
Solo la logística híbrida aprende, así que es la única que puede sufrir al cambiar de población. Para cada dirección (origen → destino) se entrena con C2 + C3 del grupo de origen, se calibra con Platt en C4 del grupo de origen y se prueba en C4 del grupo de destino. Se compara con el modelo de la Fase 2 (entrenado con todos). Los grupos con menos de 5 casos no se muestran.

In [ ]:
# @title Entrenar con un subgrupo y probar con otro
CAMBIO = es.escenario_cambio(ENTRENAMIENTO, C4, PARAMS, SEMILLA, MIN)
display(CAMBIO)

## Figuras, límites operativos y conclusiones
Un nivel de perturbación es **útil** si el Lift@k de la regla D2 sigue en 1,20 o más (la meta del proyecto y el umbral de alerta), tanto en la media como en el percentil 5 de las repeticiones.

In [ ]:
# @title Figuras, límites y conclusiones
import matplotlib.pyplot as plt
from src import graficos_escenarios as ge

RESULTADOS = pd.concat([RUIDO, FALTANTES, DESBALANCE, CAIDA], ignore_index=True)
fig = ge.figura_degradacion(RESULTADOS, "C4", UMBRAL)
ge.guardar(fig, FIG / f"{FUENTE}_29_escenarios_criticos.png", DPI); plt.show()
if not CAMBIO.empty:
    fig = ge.figura_cambio(CAMBIO)
    ge.guardar(fig, FIG / f"{FUENTE}_30_cambio_distribucion.png", DPI); plt.show()

LIMITES = es.limites(RESULTADOS, UMBRAL)
display(LIMITES)
CONCLUSIONES = es.conclusiones(RESULTADOS, CAMBIO, LIMITES, UMBRAL)
for frase in CONCLUSIONES:
    print("•", frase)

RESULTADOS.to_csv(MET / f"escenarios_criticos_{FUENTE}.csv", index=False, float_format="%.4f")
CAMBIO.to_csv(MET / f"escenarios_cambio_{FUENTE}.csv", index=False, float_format="%.4f")
LIMITES.to_csv(MET / f"escenarios_limites_{FUENTE}.csv", index=False)
resumen = {"fuente": FUENTE, "cohortes_entrenamiento": COHORTES, "perturbada": "C4",
           "k_por_sede": K, "repeticiones": REPETICIONES, "umbral_lift": UMBRAL,
           "parametros_hibrido": PARAMS, "comprobacion_sin_perturbar": COMPROBACION,
           "conclusiones": CONCLUSIONES}
(MET / f"escenarios_criticos_{FUENTE}.json").write_text(
    json.dumps(resumen, indent=2, ensure_ascii=False, default=float), encoding="utf-8")
print("\nGuardado: escenarios_criticos, escenarios_cambio y escenarios_limites")

## Qué hace el sistema cuando deja de ser útil
- **Al generar la lista**, la aplicación revisa las señales del ciclo cargado (`inferencia.revisar_senales`): si casi todos figuran como nuevos (falta la hoja anterior), si falta la fecha de pago en más del 20 % de los que no son nuevos, si los atrasos llegan vacíos o todos en cero, o si el tipo de reserva está vacío. En esos casos muestra un aviso en rojo y pide corregir el Excel antes de usar la lista. Los umbrales están en `config.yaml` (`calidad_senales`) y se revisan con los límites de este cuaderno.
- **Cada mayo**, con el resultado de la campaña, se recalcula el Lift@k de la lista. Si baja de 1,20, se activa la alerta y quien designe la Dirección decide si se suspende el uso (decisión 5 de la presentación a la Dirección).
- Detalle en `docs/escenarios_criticos.md`.

In [ ]:
# @title Descargar agregados para compartir (Colab)
# Solo métricas y figuras agregadas: no contiene datos individuales.
import shutil
paquete = RAIZ / f"escenarios_{FUENTE}"
shutil.rmtree(paquete, ignore_errors=True); paquete.mkdir()
for nombre in [f"escenarios_criticos_{FUENTE}.csv", f"escenarios_cambio_{FUENTE}.csv",
               f"escenarios_limites_{FUENTE}.csv", f"escenarios_criticos_{FUENTE}.json"]:
    shutil.copy(MET / nombre, paquete)
for png in sorted(FIG.glob(f"{FUENTE}_29_*.png")) + sorted(FIG.glob(f"{FUENTE}_30_*.png")):
    shutil.copy(png, paquete)
zip_final = shutil.make_archive(str(paquete), "zip", paquete)
shutil.rmtree(paquete)
print("Contenido:", sorted(zipfile.ZipFile(zip_final).namelist()))
if EN_COLAB:
    from google.colab import files
    files.download(zip_final)